# Imports

In [25]:
source("local-functions.R")
source("limesurvey_add_functions.R")

# Study 1. Participants Responses

In [26]:
df1 = haven::read_sav("../data/Pilot1.sav") |> add_participation() |> dplyr::filter(participation == "Participated")

93 people accepted the follow up UBT. The rest of participants (578 out of 671) didn't answer the question (G05Q18) why they didn't accept the UBT. They were probably tired of so many similar questions.

In [27]:
df1 = df1 |> add_willingness(1) |> 
            add_gender() |> add_education() |> add_health() |> 
            add_family_history(1) |> add_awareness_GC(1) |> add_country(1) |>
            add_knowing() |> add_awareness_pilory() |> add_prior_pilory() |>
            add_info_testing_procedure(1) |> add_info_urea_test() |> add_info_followup() |> 
            add_info_further_treatment() |> add_quality_info(1) |> add_postblood_test_explanation() |>
            add_blood_test_result() |> add_accepted_followup_UBT() |> 
            add_experience_followup_UBT() |> add_promptly_UBT() |> 
            add_treatment_explanation_findings(1) |> add_UBT_result() |>
            add_agreed_treatment() |> add_experience_treatment_procedure(1) |>
            add_promptly_treatment() |> add_willing_formore_tests(1) |> 
            add_appropriately_contacted()


## Counts

In [28]:
categorical_fields = c("Gender"="gender",
                       "Education"="education",
                       "Health"="health",
                       "Has any member of your family been diagnosed with cancer?"="family_history",
                       "Do you know any of the risk factors and warning symptoms of stomach cancer?"="awareness_GC",
                       "Do you personally know anyone who has had H. pylori infection?"="knowing",
                       "Are you aware of the possible consequences of H. pylori infection?"="awareness_pilory",
                       "Have you ever been tested for H. pylori before participating in this study?"="prior_pilory",
                       "Enough information about undergoing testing procedure"="info_testing_procedure", 
                       "Enough information about the necessity of a UBT"="info_urea_test",
                       "Enough information about the necessity of a follow up UBT"="info_followup", 
                       "Enough information about the necessity of further treatment"="info_further_treatment", 
                       "How do you qualify all information "="quality_info", 
                       "How did you find the explanation of the findings?"="postblood_test_explanation",
                       "Was your blood test positive?"="blood_test_result", 
                       "Did you accept a follow-up Urea Breath Test?"="accepted_followup_UBT", 
                       "How was your experience of the follow-up Urea Breath Test?"="experience_followup_UBT", 
                       "Did you find that the follow-up Urea Breath Test was provided promptly/on time?"="promptly_UBT", 
                       "After the Urea Breath Test, how did you find the explanation of the findings?"="treatment_explanation_findings", 
                       "Was your Urea Breath test positive?"="UBT_result", 
                       "Did you agree to undergo treatment?"="agreed_treatment", 
                       "How was your experience of the treatment procedure?"="experience_treatment_procedure", 
                       "Did you find that the treatment was provided to you promptly/on time?"="promptly_treatment", 
                       "Would you be willing to undergo additional treatement/analysis?"="willing_formore_tests", 
                       "Did you find the way you were contacted about the study appropriate?"="appropriately_contacted")

for (heading in names(categorical_fields)) {
  variable <- categorical_fields[[heading]]
  # Display heading
  IRdisplay::display_markdown(
    paste0("### ", heading, "  (versus willingness)")
  )
  # Generate and display frequency table
  table <-  df1 |> janitor_table("willingness_bin", variable) |>  janitor::adorn_totals("col")
  IRdisplay::display_html(
    as.character(
        knitr::kable(
          table,
          format = "html",
          align = "lrr"
        )
    )
  )
}

### Gender  (versus willingness)

willingness_bin,Female,Male,Diverse/Non-binary,I prefer not to answer,Didn't answer the question,Total
willing,411,232,2,1,0,646
not willing,13,6,0,0,0,19
Didn't answer,4,2,0,0,0,6
Total,428,240,2,1,0,671


### Education  (versus willingness)

willingness_bin,Higher education (bachelor’s degree or higher),High school or vocational,Primary/did not finish high school,I prefer not to answer,Didn't answer the question,Total
willing,568,72,3,1,2,646
not willing,11,7,1,0,0,19
Didn't answer,4,1,1,0,0,6
Total,583,80,5,1,2,671


### Health  (versus willingness)

willingness_bin,Excellent,Good,Neither good or bad,Bad,Very bad,I don't know,I prefer not to respond,Didn't answer the question,Total
willing,221,370,50,3,0,1,0,1,646
not willing,5,10,3,0,0,1,0,0,19
Didn't answer,0,3,3,0,0,0,0,0,6
Total,226,383,56,3,0,2,0,1,671


... (40 outputs left out)

### Would you be willing to undergo additional treatement/analysis?  (versus willingness)

willingness_bin,Most likely,Likely,Unsure,Unlikely,Very unlikely,I prefer not to answer,Didn't answer the question,Total
willing,517,99,9,6,1,0,14,646
not willing,7,5,4,3,0,0,0,19
Didn't answer,1,0,0,0,0,0,5,6
Total,525,104,13,9,1,0,19,671


### Did you find the way you were contacted about the study appropriate?  (versus willingness)

willingness_bin,Yes,Other,I don't know,I prefer not to answer,Didn't answer the question,Total
willing,612,10,12,1,11,646
not willing,16,1,0,2,0,19
Didn't answer,1,0,0,0,5,6
Total,629,11,12,3,16,671


In [29]:
# Source column names
variables <- c("G04Q16_SQ001", "G04Q16_SQ002",
  paste0("G04Q16_SQ", sprintf("%03d", 5:11)),
  "G04Q16_other"
)

# Descriptive labels
labels <- c(
  "Wanted to know if I had an H. pylori infection",
  "Wanted to have my H. pylori infection treated if I had it",
  "Testing was free of charge",
  "Trusted the healthcare system/my healthcare provider",
  "Had symptoms that might be related to H. pylori",
  "Wanted to reduce my risk for gastric cancer",
  "Testing procedures didn't sound uncomfortable",
  "Testing procedures sounded non-invasive",
  "Didn't undergo a blood test",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df1$G04Q16_other[
  !is.na(df1$G04Q16_other) &
    trimws(df1$G04Q16_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df1[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why wanting an H. pylori test"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why wanting an H. pylori test

,Wanted to know if I had an H. pylori infection,Wanted to reduce my risk for gastric cancer,Testing was free of charge,Wanted to have my H. pylori infection treated if I had it,Trusted the healthcare system/my healthcare provider,Testing procedures sounded non-invasive,Testing procedures didn't sound uncomfortable,Had symptoms that might be related to H. pylori,Didn't undergo a blood test,Other
Wanted to know if I had an H. pylori infection,556,345,307,270,196,194,182,86,2,19
Wanted to reduce my risk for gastric cancer,345,365,250,237,166,176,166,65,2,13
Testing was free of charge,307,250,331,193,152,165,153,60,1,11
Wanted to have my H. pylori infection treated if I had it,270,237,193,284,148,146,137,56,2,11
Trusted the healthcare system/my healthcare provider,196,166,152,148,214,114,103,33,0,11
Testing procedures sounded non-invasive,194,176,165,146,114,207,154,32,2,10
Testing procedures didn't sound uncomfortable,182,166,153,137,103,154,188,31,1,9
Had symptoms that might be related to H. pylori,86,65,60,56,33,32,31,94,1,4
Didn't undergo a blood test,2,2,1,2,0,2,1,1,41,0
Other,19,13,11,11,11,10,9,4,0,30


In [30]:
# Source column names
variables <- c(paste0("G06Q36_SQ", sprintf("%03d", 1:6)), "G06Q36_other")

# Descriptive labels
labels <- c(
  "I wanted to have my H. pylori infection treated",
  "I wanted to reduce my risk for gastric cancer",
  "I have been having symptoms that might be related to H. pylori",
  "Taking antibiotics is not burdensome",
  "I trust the healthcare system/my healthcare provider",
  "This treatment is free of charge",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df1$G06Q36_other[
  !is.na(df1$G06Q36_other) &
    trimws(df1$G04Q16_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df1[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why agreeing to antibiotics treatment?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why agreeing to antibiotics treatment?

,I wanted to have my H. pylori infection treated,I wanted to reduce my risk for gastric cancer,Other,I trust the healthcare system/my healthcare provider,This treatment is free of charge,I have been having symptoms that might be related to H. pylori,Taking antibiotics is not burdensome
I wanted to have my H. pylori infection treated,47,38,1,25,18,14,9
I wanted to reduce my risk for gastric cancer,38,40,1,25,18,14,10
Other,1,1,30,1,1,0,0
I trust the healthcare system/my healthcare provider,25,25,1,26,13,7,8
This treatment is free of charge,18,18,1,13,19,6,6
I have been having symptoms that might be related to H. pylori,14,14,0,7,6,15,6
Taking antibiotics is not burdensome,9,10,0,8,6,6,10


# Study 2. Participants Responses

In [31]:
df2 = haven::read_sav("../data/Pilot2.sav") |> add_participation() |> dplyr::filter(participation == "Participated")

In [32]:
df2 = df2 |> add_willingness(2) |> add_age() |> 
            add_gender() |> add_education() |> add_health() |> 
            add_family_history(2) |> add_awareness_GC(2) |> add_country(2) |>
            add_prior_endoscopy() |> add_prior_GC_testing() |>
            add_info_testing_procedure(2) |> add_quality_info(2) |> 
            add_treatment_explanation_findings(2) |> add_experience_treatment_procedure(2) |> 
            add_willing_formore_tests(2) 


## Counts

In [33]:
categorical_fields = c("Age"="age",
                       "Gender"="gender",
                       "Education"="education",
                       "Health"="health",
                       "Has any member of your family been diagnosed with cancer?"="family_history",
                       "Do you know any of the risk factors and warning symptoms of stomach cancer?"="awareness_GC",
                       "Have you had an upper endoscopy to checkup your stomach (camera test of the stomach) before?"="prior_endoscopy",
                       "Did you receive enough information about undergoing two procedures (endoscopy and colonoscopy) in the same session?"="info_testing_procedure", 
                       "How do you qualify all information (written or verbally) provided before upper endoscopy and colonoscopy?"="quality_info", 
                       "After the performed procedures, how did you find the explanation of the procedures and the findings?"="treatment_explanation_findings", 
                       "Did you experience pain/discomfort after the upper endoscopy and colonoscopy?"="experience_treatment_procedure", 
                       "If there is a need for additional treatment or tests and follow-up analysis after the upper endoscopy, would you be willing to undergo them?"="willing_formore_tests")

for (heading in names(categorical_fields)) {
  variable <- categorical_fields[[heading]]
  # Display heading
  IRdisplay::display_markdown(
    paste0("### ", heading, "  (versus willingness)")
  )
  # Generate and display frequency table
  table <-  df2 |> janitor_table("willingness_bin", variable) |>  janitor::adorn_totals("col")
  IRdisplay::display_html(
    as.character(
        knitr::kable(
          table,
          format = "html",
          align = "lrr"
        )
    )
  )
}

### Age  (versus willingness)

willingness_bin,Didn't answer the question,50-54,55-59,60-64,65-69,70+,I prefer not to answer,Total
Didn't answer,4,6,7,6,8,6,0,37
not willing,2,6,22,14,32,21,0,97
willing,12,174,193,232,199,135,0,945
Total,18,186,222,252,239,162,0,1079


### Gender  (versus willingness)

willingness_bin,Female,Male,Diverse/Non-binary,I prefer not to answer,Didn't answer the question,Total
Didn't answer,17,16,0,0,4,37
not willing,45,51,0,0,1,97
willing,489,449,0,0,7,945
Total,551,516,0,0,12,1079


### Education  (versus willingness)

willingness_bin,Higher education (bachelor’s degree or higher),High school or vocational,Primary/did not finish high school,I prefer not to answer,Didn't answer the question,Total
Didn't answer,3,10,10,2,12,37
not willing,20,42,25,6,4,97
willing,314,510,105,7,9,945
Total,337,562,140,15,25,1079


... (14 outputs left out)

### Did you experience pain/discomfort after the upper endoscopy and colonoscopy?  (versus willingness)

willingness_bin,No pain/discomfort,Slight pain/discomfort,Moderate pain/discomfort,Severe pain/discomfort,I don't know,I prefer not to answer,Didn't answer the question,Total
Didn't answer,18,7,5,0,0,0,7,37
not willing,72,16,4,0,2,2,1,97
willing,729,164,35,8,3,0,6,945
Total,819,187,44,8,5,2,14,1079


### If there is a need for additional treatment or tests and follow-up analysis after the upper endoscopy, would you be willing to undergo them?  (versus willingness)

willingness_bin,I certainly would,I most likely would,I likely would,I’m unsure,It is unlikely that I would,It is very unlikely that I would,I prefer not to answer,Didn't answer the question,Total
Didn't answer,14,5,3,0,0,0,1,14,37
not willing,26,25,29,8,3,0,6,0,97
willing,570,278,83,8,2,2,0,2,945
Total,610,308,115,16,5,2,7,16,1079


In [34]:
# Source column names
variables <- c(paste0("G03Q15_SQ", sprintf("%03d", 1:5)), "G03Q15_other")

# Descriptive labels
labels <- c(
  "No symptoms",
  "Fear of undergoing endoscopy",
  "Do not know the benefits of screening",
  "Financial limitations",
  "I have not been invited to a screening before",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df2$G03Q15_other[
  !is.na(df2$G03Q15_other) &
    trimws(df2$G03Q15_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df2[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why did you not undergo stomach cancer testing before?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why did you not undergo stomach cancer testing before?

,No symptoms,I have not been invited to a screening before,Do not know the benefits of screening,Fear of undergoing endoscopy,Other,Financial limitations
No symptoms,553,114,44,5,1,1
I have not been invited to a screening before,114,410,117,9,1,5
Do not know the benefits of screening,44,117,138,3,0,3
Fear of undergoing endoscopy,5,9,3,28,0,1
Other,1,1,0,0,15,0
Financial limitations,1,5,3,1,0,6


In [35]:
# Source column names
variables <- c(paste0("G05Q18_SQ", sprintf("%03d", 1:10)), "G05Q18_other")

# Descriptive labels
labels <- c(
  "Upper Endoscopy sounds non-invasive",
  "I would want to know if I had stomach cancer",
  "Upper Endoscopy does not sound uncomfortable",
  "I have stomach problems",
  "I have a family history of stomach cancer",
  "I have had (or currently have) H. pylori infection",  
  "My physician suggested it", 
  "I prefer to be checked",
  "It’s free of charge",
  "I prefer to undergo the two procedures in the same day",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df2$G05Q18_other[
  !is.na(df2$G05Q18_other) &
    trimws(df2$G05Q18_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df2[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why did you accept an upper endoscopy in the same session as a colonoscopy?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why did you accept an upper endoscopy in the same session as a colonoscopy?

,I prefer to undergo the two procedures in the same day,It’s free of charge,My physician suggested it,I would want to know if I had stomach cancer,I have stomach problems,Other,I prefer to be checked,I have had (or currently have) H. pylori infection,Upper Endoscopy does not sound uncomfortable,I have a family history of stomach cancer,Upper Endoscopy sounds non-invasive
I prefer to undergo the two procedures in the same day,541,197,106,84,38,13,40,32,28,21,17
It’s free of charge,197,348,61,66,26,6,40,15,19,15,10
My physician suggested it,106,61,284,59,22,1,19,16,8,9,8
I would want to know if I had stomach cancer,84,66,59,225,33,3,28,13,11,22,14
I have stomach problems,38,26,22,33,81,0,7,12,1,11,5
Other,13,6,1,3,0,63,2,1,1,0,0
I prefer to be checked,40,40,19,28,7,2,56,4,10,3,6
I have had (or currently have) H. pylori infection,32,15,16,13,12,1,4,51,1,6,5
Upper Endoscopy does not sound uncomfortable,28,19,8,11,1,1,10,1,48,1,8
I have a family history of stomach cancer,21,15,9,22,11,0,3,6,1,47,1
